# 04 — Preparação de Dados, Engenharia de Atributos e Pipelines

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/04_preparacao_pipelines.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_preparacao_pipelines.pdf)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Tratar ausências, categorias, escalas e engenharia de atributos sem vazamento.

`ColumnTransformer` aplica tratamentos por coluna; `Pipeline` conecta transformações a modelo. Colunas categóricas podem conter valores novos no teste, então `OneHotEncoder(handle_unknown="ignore")` é mais robusto do que mapear categorias manualmente. Seleção de atributos e reamostragem, quando existem, também devem ocorrer apenas dentro dos folds de treino.


## 2. Fundamentos matemáticos

Padronização: $z=(x-\mu_{\mathrm{treino}})/\sigma_{\mathrm{treino}}$. O uso de $\mu$ ou $\sigma$ estimados sobre todos os dados vaza informações do teste. Para variáveis categóricas, uma representação *one-hot* é um vetor indicador por categoria conhecida.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
rng=np.random.default_rng(42);n=180
X=pd.DataFrame({'idade':rng.normal(42,12,n),'renda':rng.normal(3000,900,n),'canal':rng.choice(['online','loja'],n)})
X.loc[rng.choice(n,18,replace=False),'renda']=np.nan
y=((X['idade']>45) & (X['canal']=='online')).astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,stratify=y,test_size=.25,random_state=42)
num=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
prep=ColumnTransformer([('numericas',num,['idade','renda']),('categoricas',cat,['canal'])])
model=Pipeline([('prep',prep),('classificador',LogisticRegression(max_iter=1000))]).fit(Xtr,ytr)
print('Acurácia balanceada teste:',round(balanced_accuracy_score(yte,model.predict(Xte)),3))

Acurácia balanceada teste: 0.938


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
# Demonstrar transformação de categoria não vista sem dar fit no teste
novo=pd.DataFrame({'idade':[34.], 'renda':[3200.], 'canal':['telefone']})
print('Previsão com categoria desconhecida:',model.predict(novo))
print('Nomes de atributos:',model.named_steps['prep'].get_feature_names_out().tolist())

Previsão com categoria desconhecida: [0]
Nomes de atributos: ['numericas__idade', 'numericas__renda', 'categoricas__canal_loja', 'categoricas__canal_online']


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: verificar que imputador numérico aprende só no treino
learned=model.named_steps['prep'].named_transformers_['numericas'].named_steps['imputer'].statistics_
print('Medianas estimadas apenas no treino:',learned.round(2))

Medianas estimadas apenas no treino: [  41.35 2939.74]


## 6. Interpretação, limitações e erros comuns

Não confundir imputação com recuperação do valor verdadeiro. *Encoding* ordinal só é adequado quando as categorias realmente possuem ordem, ou quando o algoritmo lida corretamente com a codificação. Pipelines melhoram reprodutibilidade e segurança metodológica; não garantem, por si sós, ausência de vazamento na construção original das colunas.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- Fazer split ANTES de `fit`/`fit_transform` de qualquer transformação aprendida.
- `Pipeline` dentro de `cross_val_score` reestima transformadores a cada fold.
- Usar `ColumnTransformer` para dados heterogêneos.
- Validar valores desconhecidos e colunas ausentes.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 10-preparacao-de-dados-1.pdf; 10-preparacao-de-dados-2.pdf; Disciplina 02 — ETL.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** pipeline dentro do treino. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **pipeline dentro do treino** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Pré-processamento sem vazamento

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.


In [4]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
X_treino = np.array([[1., 10.], [2., np.nan], [4., 20.]])
X_teste = np.array([[100., np.nan]])
p = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
p.fit(X_treino)
print("Treino transformado:", p.transform(X_treino))
print("Teste transformado sem novo fit:", p.transform(X_teste))

Treino transformado: [[-1.06904497 -1.22474487]
 [-0.26726124  0.        ]
 [ 1.33630621  1.22474487]]
Teste transformado sem novo fit: [[78.30754388  0.        ]]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **pipeline dentro do treino** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
